# MSD Default Post-Processing Diagnostic

This notebook reproduces the diagnostic run comparing the noiseless Steane/Gemini ancilla observable branches obtained through `default_post_processing(reg)` against candidate factory targets such as `1011` and `0000`.


In [1]:
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not locate repo root containing demo/msd_utils.')

from bloqade.lanes import GeminiLogicalSimulator
from demo.msd_utils.circuits import (
    build_decoder_kernel_bundle,
    build_task_map,
    make_noisy_steane7_initializer,
)
from demo.msd_utils.core import logical_expectation, run_task


In [2]:
THETA = 0.3041 * np.pi
PHI = 0.25 * np.pi
LAM = 0.0
SHOTS = 20_000


## Build Steane/Gemini tasks using `default_post_processing`

These kernels already return `default_post_processing(reg)`, so we intentionally do **not** append matrix-based detector/observable annotations here.

In [3]:
sim = GeminiLogicalSimulator()
noisy_init = make_noisy_steane7_initializer(sim)
kernel_bundle = build_decoder_kernel_bundle(THETA, PHI, LAM, output_qubit=0)

actual_tasks = build_task_map(
    sim,
    kernel_bundle.actual,
    m2dets=None,
    m2obs=None,
    noisy_initializer=noisy_init,
    append_measurements=False,
)


In [4]:
x_data = run_task(actual_tasks['X'], SHOTS, with_noise=False)
y_data = run_task(actual_tasks['Y'], SHOTS, with_noise=False)
z_data = run_task(actual_tasks['Z'], SHOTS, with_noise=False)

print('Noiseless raw <X,Y,Z> from default_post_processing path =', (
    logical_expectation(x_data.observables[:, 0]),
    logical_expectation(y_data.observables[:, 0]),
    logical_expectation(z_data.observables[:, 0]),
))


Noiseless raw <X,Y,Z> from default_post_processing path = (0.1932, -0.1901, 0.2005)


In [5]:
def top_branches(data, shots=SHOTS):
    counts = {}
    for row in data.observables[:, 1:]:
        key = tuple(int(x) for x in row)
        counts[key] = counts.get(key, 0) + 1
    return [(key, value / shots) for key, value in sorted(counts.items(), key=lambda kv: kv[1], reverse=True)]

for basis, data in [('X', x_data), ('Y', y_data), ('Z', z_data)]:
    print(f'\nTop noiseless ancilla branches for {basis}:')
    for pattern, frac in top_branches(data)[:8]:
        print(f'  {pattern}: {frac:.5f}')



Top noiseless ancilla branches for X:
  (0, 0, 0, 0): 0.16840
  (1, 0, 1, 1): 0.05945
  (1, 1, 0, 1): 0.05730
  (0, 0, 0, 1): 0.05695
  (0, 0, 1, 1): 0.05645
  (1, 1, 1, 1): 0.05605
  (1, 1, 1, 0): 0.05600
  (0, 1, 0, 1): 0.05580

Top noiseless ancilla branches for Y:
  (0, 0, 0, 0): 0.15825
  (1, 1, 1, 0): 0.11970
  (1, 0, 1, 0): 0.07540
  (0, 0, 1, 0): 0.07360
  (0, 1, 0, 0): 0.06960
  (0, 1, 0, 1): 0.05760
  (1, 1, 0, 1): 0.05735
  (1, 0, 0, 0): 0.05590

Top noiseless ancilla branches for Z:
  (0, 0, 0, 0): 0.16570
  (0, 0, 1, 1): 0.06995
  (0, 1, 0, 0): 0.06670
  (1, 0, 0, 1): 0.06660
  (0, 1, 1, 0): 0.05695
  (0, 1, 1, 1): 0.05675
  (1, 1, 1, 1): 0.05550
  (1, 1, 0, 1): 0.05525


In [7]:
def summarize_target(factory_target):
    target = np.asarray(factory_target, dtype=np.uint8)
    print(f'\nTarget {tuple(int(x) for x in target)}')
    for basis, data in [('X', x_data), ('Y', y_data), ('Z', z_data)]:
        mask = np.all(data.observables[:, 1:] == target, axis=1)
        bits = data.observables[mask, 0].astype(np.uint8)
        exp = float('nan') if len(bits) == 0 else logical_expectation(bits)
        print(
            f'  {basis}: accepted_fraction={mask.mean():.5f}, raw_expectation={exp:.6f}, shots={len(bits)}'
        )

for candidate in [
    np.array([1, 0, 1, 1], dtype=np.uint8),
    np.array([0, 0, 0, 0], dtype=np.uint8),
]:
    summarize_target(candidate)



Target (1, 0, 1, 1)
  X: accepted_fraction=0.05945, raw_expectation=0.597981, shots=1189
  Y: accepted_fraction=0.04980, raw_expectation=-0.357430, shots=996
  Z: accepted_fraction=0.05405, raw_expectation=0.587419, shots=1081

Target (0, 0, 0, 0)
  X: accepted_fraction=0.16840, raw_expectation=0.567696, shots=3368
  Y: accepted_fraction=0.15825, raw_expectation=-0.862875, shots=3165
  Z: accepted_fraction=0.16570, raw_expectation=0.585999, shots=3314


## Notes

If this notebook still shows `0000` as the dominant ancilla branch in the Steane/Gemini observable frame while the bare 5-qubit logical MSD circuit prefers `1011`, then the mismatch is not coming from `append_measurements_and_annotations(...)`. It is coming from the encoded Steane/Gemini readout convention itself.